# 04 - Scanner Threshold Recommendations

Derives initial scanner thresholds from normalized outputs and produces recommendation tables/charts.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
from pathlib import Path
import pandas as pd
import plotly.express as px

from src.common import load_config

cfg = load_config('../config/exploration.yaml')
processed_dir = cfg.outputs_processed_dir
files = list(Path(processed_dir).glob('*polygon_snapshot.parquet'))
assert files, 'Run notebook 01 first to generate polygon snapshot parquet'

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
df = df.dropna(subset=['change_pct','volume'])

df.head()


In [ ]:
pct_candidates = [2, 3, 4, 5, 6, 8, 10]
vol_candidates = [50000, 100000, 200000, 500000, 1000000]

rows = []
for p in pct_candidates:
    for v in vol_candidates:
        matched = df[(df['change_pct'] >= p) & (df['volume'] >= v)]
        rows.append({
            'min_change_pct': p,
            'min_volume': v,
            'match_count': len(matched),
            'median_change_pct': matched['change_pct'].median() if not matched.empty else None,
            'median_volume': matched['volume'].median() if not matched.empty else None,
        })

df_grid = pd.DataFrame(rows)
df_grid.sort_values(['min_change_pct','min_volume']).head(30)


In [ ]:
pivot = df_grid.pivot(index='min_volume', columns='min_change_pct', values='match_count')
fig = px.imshow(pivot, text_auto=True, title='Candidate Threshold Match Counts')
fig.update_layout(xaxis_title='min_change_pct', yaxis_title='min_volume')
fig.show()


In [ ]:
recommended = df_grid[(df_grid['min_change_pct'] == 4) & (df_grid['min_volume'] == 100000)]
print('Baseline recommendation (initial):')
display(recommended)

session_rec = (
    df.groupby('session')[['change_pct','volume']]
      .quantile([0.5, 0.75, 0.9])
      .reset_index()
      .rename(columns={'level_1':'quantile'})
)
session_rec
